# Plot TF-activity results — AbbasTFScreen

All four parts = OLS regression of peak Log2FC on the grouped HOCOMOCO motif matrix:
- **Part A — per cell state** (`StateTFActivity_*`): each cell state vs the rest. Well-powered.
- **Part B — per state × perturbation** (`TFActivity_*_stateXpert`): each perturbation vs same-state NTC. Sparse.
- **Part C — across pseudotime** (`PseudotimeTFActivity_*`): each pseudotime segment vs the root (NE) segment along the DPT trajectory — which TFs gain/lose activity NE → Differentiated.
- **Part D — per perturbation, pooled** (`PertTFActivity_ptpeaks_*`): each perturbation vs pooled NTC, no state stratification, on the pseudotime peak set.

Coefficient > 0 = TF's motif-peaks open ⇒ **more active**; < 0 = **less active**. `*`/shown cells pass FDR ≤ 0.1.

In [ ]:
suppressPackageStartupMessages({
    library(pheatmap); library(ggplot2); library(data.table); library(RColorBrewer)
})
CSV <- "/home/eraslab1/Projects/AbbasTFScreen/CSV_Files"
FDR <- 0.1

## Part A — TF activity per cell state
Heatmap of TFs significant in ≥1 state; states ordered along the NE→Differentiated trajectory; `*` = FDR ≤ 0.1.

In [ ]:
coef <- as.matrix(fread(file.path(CSV,"StateTFActivity_coef.csv")), rownames=1)
fdr  <- as.matrix(fread(file.path(CSV,"StateTFActivity_FDR.csv")),  rownames=1)

# --- states in trajectory order: Neuroendocrine -> Differentiated (edit if needed).
#     pseudotime: NE .069, Int-3 .065, Int-1 .137, Int-2 .154, Diff-2 .237, Diff_1 .431
STATE_ORDER <- c("Neuroendocrine","Intermediate-3","Intermediate-1","Intermediate-2",
                 "Differentiated-2","Differentiated_1")
STATE_ORDER <- intersect(STATE_ORDER, colnames(coef))
coef <- coef[, STATE_ORDER]; fdr <- fdr[, STATE_ORDER]

# TFs significant (FDR<=0.1) in >=1 state
sigTF <- rownames(coef)[rowSums(fdr <= FDR, na.rm=TRUE) >= 1]
mat  <- coef[sigTF, , drop=FALSE]
star <- ifelse(fdr[sigTF, , drop=FALSE] <= FDR, "*", "")

# --- order TF rows by WHERE they peak along the trajectory, so the heatmap runs
#     top -> bottom as Neuroendocrine-active -> Differentiated-active ---
peak    <- max.col(mat, ties.method="first")                  # state of each TF's max activity
row_ord <- order(peak, -mat[cbind(seq_len(nrow(mat)), peak)]) # then strongest first within a state
mat  <- mat[row_ord, , drop=FALSE]
star <- star[row_ord, , drop=FALSE]
cat(length(sigTF), "sig TFs |  states:", paste(STATE_ORDER, collapse=" -> "), "\n")

In [ ]:
lim <- max(abs(mat))
options(repr.plot.width=8, repr.plot.height=max(5, 0.18*nrow(mat)))
pheatmap(mat,
         cluster_cols = FALSE,      # states kept in trajectory order (NE -> Differentiated)
         cluster_rows = FALSE,      # TFs ordered by where they peak along the trajectory
         display_numbers = star, number_color = "black", fontsize_number = 11,
         color  = colorRampPalette(c("#2166AC","white","#B2182B"))(100),
         breaks = seq(-lim, lim, length.out = 101),
         fontsize_row = 7, fontsize_col = 10, angle_col = 90,
         main = "TF activity per cell state  (NE -> Differentiated;  * FDR<=0.1)")

## Part B — TF activity per (state × perturbation)
This is the sparse analysis (56 significant calls). Three views: counts per state, the nonzero sub-matrix, and the calls faceted by state.

In [ ]:
sig <- fread(file.path(CSV,"TFActivity_significant_long.csv"))
cat(nrow(sig), "significant calls |",
    length(unique(paste(sig$State, sig$Perturbation))), "conditions |",
    length(unique(sig$TF)), "TFs\n")

# (1) counts per state x direction
options(repr.plot.width=7, repr.plot.height=4)
ggplot(sig, aes(State, fill=direction)) +
    geom_bar(position="dodge") +
    scale_fill_manual(values=c(more_active="#B2182B", less_active="#2166AC")) +
    theme_bw() + theme(axis.text.x=element_text(angle=30,hjust=1)) +
    labs(title="Significant TF-activity calls per state (state x perturbation)", y="# calls")

In [ ]:
# (2) heatmap of the nonzero sub-matrix of the FDR-significant coefficients
cf <- as.matrix(fread(file.path(CSV,"TFActivity_coefFDRsig_stateXpert.csv")), rownames=1)
cf <- cf[rowSums(cf!=0)>0, colSums(cf!=0)>0, drop=FALSE]   # keep TFs/conditions with a hit
cat("nonzero sub-matrix:", nrow(cf), "TFs x", ncol(cf), "conditions\n")
lim2 <- max(abs(cf))
options(repr.plot.width=max(6,0.22*ncol(cf)), repr.plot.height=max(5,0.22*nrow(cf)))
pheatmap(cf,
         color  = colorRampPalette(c("#2166AC","white","#B2182B"))(100),
         breaks = seq(-lim2, lim2, length.out=101),
         fontsize_row=6, fontsize_col=6,
         main="state x perturbation: FDR-significant TF activity (nonzero only)")

In [ ]:
# (3) the significant calls, faceted by state (TF x perturbation)
options(repr.plot.width=11, repr.plot.height=8)
ggplot(sig, aes(x=Perturbation, y=TF, fill=coef)) +
    geom_tile(color="grey80") +
    facet_wrap(~State, scales="free", ncol=2) +
    scale_fill_gradient2(low="#2166AC", mid="white", high="#B2182B", midpoint=0) +
    theme_bw(base_size=8) +
    theme(axis.text.x=element_text(angle=90, hjust=1, vjust=0.5, size=6),
          axis.text.y=element_text(size=6)) +
    labs(title="Significant TF activity (FDR<=0.1): perturbation vs same-state NTC")

## Part C — TF activity across pseudotime

OLS regression of per-peak Log2FC (each pseudotime segment vs the **root** `PT_seg01`, from ArchR `getMarkerFeatures` — bias-matched, depth-normalized, exactly as in `03`) on the grouped HOCOMOCO motif matrix. Columns run along the DPT trajectory: `PT_seg01` (Neuroendocrine root) → `PT_seg16` (Differentiated tip). Rows ordered by trend — **rising** TFs at top, **falling** at bottom; `*` = FDR ≤ 0.1. Coefficient > 0 = TF's motif-peaks open along the trajectory ⇒ TF **gains activity** (e.g. ZEB1/SNAI1/SNAI2/GRHL2 rise, NFIA/NFIC fall). The line plot below shows the coefficient trajectory of the top rising/falling TFs.

In [ ]:
pcoef <- as.matrix(fread(file.path(CSV,"PseudotimeTFActivity_coef.csv")), rownames=1)
pfdr  <- as.matrix(fread(file.path(CSV,"PseudotimeTFActivity_FDR.csv")),  rownames=1)

# columns already run root(NE) -> tip(Differentiated); enforce numeric segment order
seg_ord <- colnames(pcoef)[order(as.integer(sub("PT_seg","",colnames(pcoef))))]
pcoef <- pcoef[, seg_ord]; pfdr <- pfdr[, seg_ord]

# keep TFs significant (FDR<=0.1) in >=MIN_SIG segments
MIN_SIG <- 1
sigTF <- rownames(pcoef)[rowSums(pfdr <= FDR, na.rm=TRUE) >= MIN_SIG]
pmat  <- pcoef[sigTF, , drop=FALSE]
pstar <- ifelse(pfdr[sigTF, , drop=FALSE] <= FDR, "*", "")

# order TF rows by trend along pseudotime (rising -> falling)
trend <- apply(pmat, 1, function(y) suppressWarnings(cor(seq_along(y), y, method="spearman")))
pmat  <- pmat[order(-trend), , drop=FALSE]
pstar <- pstar[order(-trend), , drop=FALSE]
cat(length(sigTF), "sig TFs across", ncol(pmat),
    "pseudotime segments (root NE -> Differentiated tip)\n")

In [ ]:
lim3 <- max(abs(pmat))
options(repr.plot.width=max(7, 0.5*ncol(pmat)), repr.plot.height=max(5, 0.16*nrow(pmat)))
pheatmap(pmat,
         cluster_cols = FALSE,   # pseudotime order: root(NE) -> tip(Differentiated)
         cluster_rows = FALSE,   # TFs ordered by trend: rising (top) -> falling (bottom)
         display_numbers = pstar, number_color="black", fontsize_number=9,
         color  = colorRampPalette(c("#2166AC","white","#B2182B"))(100),
         breaks = seq(-lim3, lim3, length.out=101),
         fontsize_row=7, fontsize_col=9, angle_col=90,
         main="TF activity across pseudotime  (root NE -> Differentiated tip;  * FDR<=0.1)")

In [ ]:
# line view: coefficient trajectory of the top rising & falling TFs across pseudotime
tr <- fread(file.path(CSV,"PseudotimeTFActivity_trend.csv"))[nSigSeg >= 1]
riseTF <- head(tr[order(-trend)]$TF, 10); fallTF <- head(tr[order(trend)]$TF, 10)
dt <- as.data.table(pcoef[c(riseTF, fallTF), , drop=FALSE], keep.rownames="TF")
dl <- melt(dt, id.vars="TF", variable.name="segment", value.name="coef")
dl[, seg_i := as.integer(sub("PT_seg","",segment))]
dl[, dir   := ifelse(TF %in% riseTF, "rising", "falling")]
options(repr.plot.width=9, repr.plot.height=5)
ggplot(dl, aes(seg_i, coef, color=TF, linetype=dir)) +
    geom_hline(yintercept=0, color="grey70") +
    geom_line(linewidth=0.8) + geom_point(size=1.2) +
    scale_x_continuous(breaks=1:max(dl$seg_i)) +
    scale_linetype_manual(values=c(rising="solid", falling="22")) +
    theme_bw() +
    labs(title="Top TFs changing across pseudotime (segment vs root)",
         x="pseudotime segment (root NE = 1 -> Differentiated tip)",
         y="regression coefficient (TF activity vs root)")

## Part D — TF activity per perturbation (pooled across states)

`PertTFActivity_ptpeaks_*`: each perturbation vs **pooled NTC** (14,024 cells), with **no cell-state stratification**, on the **pseudotime peak set** (114k peaks) + its motif matrix. `getMarkerFeatures` Log2FC (perturbation / NTC) regressed on the grouped HOCOMOCO motifs, one OLS per perturbation. Sparse — **16 significant calls across 11 of 91 perturbations** — expected when marginalizing over the dominant state/trajectory axis. Two views: significant-call counts per perturbation, and the nonzero FDR-significant coefficient sub-matrix (strongest hit: GRHL2 up under SIM1+VSX1, FDR 0.001).

In [ ]:
psig <- fread(file.path(CSV,"PertTFActivity_ptpeaks_significant_long.csv"))
cat(nrow(psig), "significant calls |",
    length(unique(psig$Perturbation)), "perturbations |",
    length(unique(psig$TF)), "TFs\n")

# significant-call counts per perturbation, colored by direction
options(repr.plot.width=7, repr.plot.height=4)
ggplot(psig, aes(reorder(Perturbation, Perturbation, length), fill=direction)) +
    geom_bar() + coord_flip() +
    scale_fill_manual(values=c(more_active="#B2182B", less_active="#2166AC")) +
    theme_bw() +
    labs(title="Significant TF-activity calls per perturbation (pooled vs NTC, pseudotime peaks)",
         x="perturbation", y="# calls")

In [ ]:
# nonzero sub-matrix of the FDR-significant coefficients (TFs x perturbations)
pcf <- as.matrix(fread(file.path(CSV,"PertTFActivity_ptpeaks_coefFDRsig.csv")), rownames=1)
pcf <- pcf[rowSums(pcf!=0)>0, colSums(pcf!=0)>0, drop=FALSE]
cat("nonzero sub-matrix:", nrow(pcf), "TFs x", ncol(pcf), "perturbations\n")
lab  <- matrix(ifelse(pcf!=0, sprintf("%.2f", pcf), ""), nrow=nrow(pcf), dimnames=dimnames(pcf))
lim4 <- max(abs(pcf))
options(repr.plot.width=max(6,0.6*ncol(pcf)), repr.plot.height=max(4,0.35*nrow(pcf)))
pheatmap(pcf,
         color  = colorRampPalette(c("#2166AC","white","#B2182B"))(100),
         breaks = seq(-lim4, lim4, length.out=101),
         display_numbers = lab, number_color="black", fontsize_number=8,
         fontsize_row=8, fontsize_col=8, angle_col=90,
         main="Perturbation vs NTC (pseudotime peaks): FDR-significant TF activity (nonzero only)")

### Notes
- Row labels like `TFGroup_N` are collapsed motif families (see `TFGroups_hocomoco.csv` for members).
- Part A is the well-powered result (clean NE→Differentiated gradient, e.g. GRHL2/ZEB1 up in Differentiated, down in Neuroendocrine). Part B is sparse/weak by comparison — expected, given the subtle within-state perturbation effects.